In [ ]:
!pip -q install pandas rouge-score bert-score sentence-transformers scikit-learn textstat
import re #for finding specific text patterns (like law paragraphs)
import os #for handling files
import math
import numpy as np
import pandas as pd

from rouge_score import rouge_scorer #measures word overlap
import bert_score #measures if the meaning is similar using AI
from sentence_transformers import SentenceTransformer #converts text to math vectors
from sklearn.metrics.pairwise import cosine_similarity #measures how close two vectors are
import textstat #checks if the readability

#import the test set
GOLD_FILE = "/content/drive/MyDrive/data/dataset_test.csv"

MODEL_FILES = {
    "Inference_model1": "/content/drive/MyDrive/data/inference_results_model1.csv",
    "FT_model2": "/content/drive/MyDrive/data/finetuned_results_model2.csv",
    "RAG_model3": "/content/drive/MyDrive/data/rag_results_model3.csv"
}

# load the test set and clean the data
gold_df = pd.read_csv(GOLD_FILE)
gold_df = gold_df[["id", "correct_answer", "sources"]].copy()
gold_df["correct_answer"] = gold_df["correct_answer"].fillna("").astype(str)
gold_df["sources"] = gold_df["sources"].fillna("").astype(str)

print(gold_df.shape)
gold_df.head(3)

# helper function to clean up text (remove extra spaces)
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

# function to find law citations like "§ 7 KStG" in the text
def extract_citations(text):
    text = normalize_text(text)

    # catches patterns like:
    # § 7 KStG
    # § 7 Abs. 1 KStG
    # § 12 Abs. 1 Z 2 KStG 1988
    pattern = r"§+\s*\d+[a-zA-Z]*?(?:\s*Abs\.\s*\d+[a-zA-Z]*)?(?:\s*Z\s*\d+[a-zA-Z]*)?(?:\s*lit\.\s*[a-z])?\s*(?:EStG|UStG|KStG|GrEStG)(?:\s*1988|\s*1994|\s*1987)?"

    matches = re.findall(pattern, text, flags=re.IGNORECASE)

    cleaned = []
    for m in matches:
        x = m.lower()
        x = re.sub(r"\s+", " ", x).strip()
        x = x.replace("  ", " ")
        x = x.replace("estg 1988", "estg")
        x = x.replace("ustg 1994", "ustg")
        x = x.replace("kstg 1988", "kstg")
        x = x.replace("grestg 1987", "grestg")
        cleaned.append(x)

    return sorted(list(set(cleaned))) #return a clean, sorted list of unique laws

# functions to check if the AI mentioned the correct laws
def citation_exact_match(pred_cites, gold_cites):
    return int(set(pred_cites) == set(gold_cites)) # 1 if perfect match, 0 if not


def citation_precision_recall_f1(pred_cites, gold_cites):
    pred_set = set(pred_cites)
    gold_set = set(gold_cites)

    if len(pred_set) == 0 and len(gold_set) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_set) == 0:
        return 0.0, 0.0, 0.0

    tp = len(pred_set & gold_set)# correct laws found
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

# check how easy the text is to read (specifically for German)
def readability_score(text):
    text = normalize_text(text)
    if text == "":
        return np.nan
    try:
        textstat.set_lang("de")
        return textstat.flesch_reading_ease(text)
    except:
        return np.nan

# combine all model results into one big table
all_results = []

for model_name, file_path in MODEL_FILES.items():
    pred_df = pd.read_csv(file_path)
    pred_df = pred_df[["id", "answer"]].copy()
    pred_df["answer"] = pred_df["answer"].fillna("").astype(str)

    merged = gold_df.merge(pred_df, on="id", how="inner")
    merged["model"] = model_name

    print(model_name, merged.shape)

    all_results.append(merged)

eval_df = pd.concat(all_results, ignore_index=True)
eval_df["correct_answer"] = eval_df["correct_answer"].apply(normalize_text)
eval_df["answer"] = eval_df["answer"].apply(normalize_text)

print(eval_df.shape)
eval_df.head(3)

#calculate ROUGE-L (how many words match the order of the perfect answer)
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

rouge_scores = []

for _, row in eval_df.iterrows():
    score = rouge.score(row["correct_answer"], row["answer"])
    rouge_scores.append(score["rougeL"].fmeasure)

eval_df["rougeL"] = rouge_scores
print("ROUGE-L done")

preds = eval_df["answer"].tolist()
refs = eval_df["correct_answer"].tolist()

# calculate BERTScore (checks if the meaning is the same, even with different words)
P, R, F1 = bert_score.score(preds, refs, lang="de", verbose=True)

eval_df["bertscore_f1"] = [x.item() for x in F1]
print("BERTScore done")

#calculate Citation Scores (did the AI cite the right laws?)
eval_df["gold_citations"] = eval_df["sources"].apply(extract_citations)
eval_df["pred_citations"] = eval_df["answer"].apply(extract_citations)

citation_em_list = []
citation_precision_list = []
citation_recall_list = []
citation_f1_list = []

for _, row in eval_df.iterrows():
    gold_cites = row["gold_citations"]
    pred_cites = row["pred_citations"]

    em = citation_exact_match(pred_cites, gold_cites)
    precision, recall, f1 = citation_precision_recall_f1(pred_cites, gold_cites)

    citation_em_list.append(em)
    citation_precision_list.append(precision)
    citation_recall_list.append(recall)
    citation_f1_list.append(f1)

eval_df["citation_em"] = citation_em_list
eval_df["citation_precision"] = citation_precision_list
eval_df["citation_recall"] = citation_recall_list
eval_df["citation_f1"] = citation_f1_list

print("Citation metrics done")

embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

ref_embeddings = embed_model.encode(eval_df["correct_answer"].tolist(), convert_to_numpy=True, show_progress_bar=True)
pred_embeddings = embed_model.encode(eval_df["answer"].tolist(), convert_to_numpy=True, show_progress_bar=True)

#calculate Cosine Similarity (checks general similarity using AI math)
cos_scores = []
for i in range(len(eval_df)):
    sim = cosine_similarity([ref_embeddings[i]], [pred_embeddings[i]])[0][0]
    cos_scores.append(float(sim))

eval_df["cosine_similarity"] = cos_scores
print("Cosine similarity done")

eval_df["readability"] = eval_df["answer"].apply(readability_score)
print("Readability done")

# group everything to see which model is the best
summary = (
    eval_df.groupby("model")
    .agg(
        BERTScore_F1=("bertscore_f1", "mean"),
        ROUGE_L=("rougeL", "mean"),
        Citation_EM=("citation_em", "mean"),
        Citation_Precision=("citation_precision", "mean"),
        Citation_Recall=("citation_recall", "mean"),
        Citation_F1=("citation_f1", "mean"),
        Cosine_Sim=("cosine_similarity", "mean"),
        Readability=("readability", "mean")
    )
    .reset_index()
)

summary = summary.sort_values("BERTScore_F1", ascending=False)
summary

#save the results to CSV files
summary.to_csv("evaluation_summary.csv", index=False)
eval_df.to_csv("evaluation_detailed.csv", index=False)

print("Saved: evaluation_summary.csv")
print("Saved: evaluation_detailed.csv")